In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

path = path + "/Q1_data.csv"
print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 6))
sns.histplot(df['Delivery_Time'])

In [ ]:
df['Order_ID']

In [ ]:
# Task 1: Write your code here:
df_no_ID = df.drop('Order_ID', axis=1)
df_no_ID

In [ ]:
# Task 2: Write your code here:
df_no_ID.isnull().sum()

In [ ]:
df_no_ID.isnull().sum().sum()

In [ ]:
df_no_ID.dropna(inplace=True)
df_no_ID.isnull().sum()

In [ ]:
# Task 3: Write your code here:
df_no_ID.duplicated().sum()

In [ ]:
# to show the exict rows that has missing values
df_no_ID[df_no_ID.duplicated()]

In [ ]:
df_no_ID.drop_duplicates(inplace=True)
df_no_ID.duplicated().sum()

In [ ]:
# Task 4: Write your code here:
df_no_ID.select_dtypes(include=['object'])

In [ ]:
categorical_cols = df_no_ID.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Unique values in column '{col}':")
    print(df_no_ID[col].unique())
    print()

In [ ]:
categorical_cols

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

for col in categorical_cols:
    print(f"Encoding column: {col}")
    encoder = LabelEncoder()
    df_no_ID[col] = encoder.fit_transform(df_no_ID[col])

In [ ]:
df_no_ID

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_no_ID.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_no_ID[numerical_cols] = scaler.fit_transform(df_no_ID[numerical_cols])
df_no_ID.head()

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df_no_ID, df_no_ID.columns)

In [ ]:
# Task 1: Write your code here:
X = df_no_ID.drop('Delivery_Time', axis=1)
y = df_no_ID['Delivery_Time']

X.shape, y.shape

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

k_folds = 5

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

In [ ]:
losses = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{k_folds}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # eval
    mae = mean_absolute_error(y_test, y_pred)
    losses.append(mae)
    print(f"MAE: {mae}")

In [ ]:
print('avrage loss over all the folds:', np.average(losses))

In [ ]:
# Task 1: Write your code here:
importances = model.feature_importances_
feature_names = X.columns

plt.figure(figsize=(10, 6))
plt.barh(feature_names, importances)
plt.xlabel("Importance")
plt.ylabel("Features")
plt.title("Feature Importance")
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
sns.histplot(y_pred, label='predicted')
sns.histplot(y_test, label='actual')
plt.legend()
plt.show()

In [ ]:
pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

k_folds = 5

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

In [ ]:
models = [RandomForestRegressor(random_state=42), CatBoostRegressor(random_state=42, verbose=0)]

losses_ensemble = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{k_folds}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  losses = []

  for model in models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    losses.append(mae)
    print(f"Model: {model.__class__.__name__}, MAE: {mae}")

  losses_ensemble.append(np.mean(losses))
  print(f"Ensemble MAE: {np.mean(losses)}")

In [ ]:
print('avrage loss over all the folds:', np.average(losses_ensemble))